## MCP File System 

### SETUP

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown
import json
from agents import trace, Runner, Agent
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX
from agents.mcp import MCPServerStdio

In [2]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

client = OpenAI()

if OPENAI_API_KEY is None:
    raise Exception("There is no openai api key")
else:
    print(OPENAI_API_KEY[:7])

sk-proj


In [3]:
# Absolute path is important: the server needs a full path for its allowed directory.
SANDBOX_DIR = os.path.abspath("secret_project_007")
os.makedirs(SANDBOX_DIR, exist_ok=True)

# Drop in a couple of sample files for the agent to find.
# encoding="utf-8" keeps the em dashes (—) safe on Windows (which defaults to cp1252).
with open(os.path.join(SANDBOX_DIR, "notes.txt"), "w", encoding="utf-8") as f:
    f.write(
        "Project Falcon — kickoff notes\n"
        "- Estimated budget: $42,000\n"
        "- Launch target: end of Q3\n"
        "- Owner: Priya\n"
        "- Core team: Marco (engineering), Lena (design), Sam (go-to-market)\n"
        "What it is:\n"
        "Falcon is a tech news and discussion community with AI avatars who provide commentary\n"
        "It's like a Hacker News + a Late Night Talk Show, but in written format and with AI commentators.\n"
        "The bet: engaging AI avatars will quickly become fan-favourites because of the unique dynamic between them.\n"
        "Premise:\n"
        "Ship before the competition announces theirs; quiet until then\n"
        "Hacker News is the benchmark — we study it constantly to understand what the community wants\n"
        "Two competitors rumored to be building something similar — speed is the moat\n"
        "'Falcon' is a placeholder; marketing hates it, but it has stuck\n"
        "Priya's one hard rule: do not slip the Q3 date to add features\n"
        "Open question: what are the most popular categories on Hacker News?\n"
    )

with open(os.path.join(SANDBOX_DIR, "todo.md"), "w", encoding="utf-8") as f:
    f.write(
        "# To do\n"
        "- [X] Book venue\n"
        "- [ ] Confirm budget with finance\n"
        "- [X] Lock the core team\n"
        "- [X] Brief the team on keeping this quiet\n"
        "- [ ] Draft announcement\n"
        "- [ ] Decide whether to keep the 'Falcon' name or rebrand before launch\n"
        "- [ ] Pressure-test the Q3 timeline with engineering\n"
        "- [ ] Scan the Hacker News front page, group top 20 stories into broad categories\n"
    )

print(f"✅ Sandbox ready at: {SANDBOX_DIR}")
print("   Files:", os.listdir(SANDBOX_DIR))

#==================================================

# Create a decoy folder
OTHER_DIR = os.path.abspath("secret_project_006")
os.makedirs(OTHER_DIR, exist_ok=True)

# Feels empty.. Drop in a decoy file.
with open(os.path.join(OTHER_DIR, "decoy_file.txt"), "w") as f:
    f.write("This is a decoy file")

print(f"✅ Decoy folder is ready at: {OTHER_DIR}")
print("   Files:", os.listdir(OTHER_DIR))

✅ Sandbox ready at: /Users/bhagwandass/Desktop/SuperDataScience/AI_Engineering_Essentials/AI Engineering Part 3/secret_project_007
   Files: ['notes.txt', 'todo.md']
✅ Decoy folder is ready at: /Users/bhagwandass/Desktop/SuperDataScience/AI_Engineering_Essentials/AI Engineering Part 3/secret_project_006
   Files: ['decoy_file.txt']


#### https://modelcontextprotocol.io/examples ---- examples from anthropic
#### https://www.pulsemcp.com/servers
#### https://mcpservers.org
#### https://github.com/modelcontextprotocol/servers/blob/599dafc1054550a6eeb87a6545c1e1b03b3ca827/src/filesystem/README.md

#### Check that Node.js + npx are available

In [4]:
!node --version
!npx --version

v24.19.0
11.17.0


### Filesystem MCP Server
#### go to the section in github where it says how to use this server with vscode

In [5]:
#defining the parameters:
filesystem_params = {
    'command' : 'npx',
    'args': ["-y",
        "@modelcontextprotocol/server-filesystem",
        SANDBOX_DIR]
}

In [6]:
#listing the tools and running the mcp server:

async with MCPServerStdio(
    name = 'Filesystem Server',
    params = filesystem_params,
    client_session_timeout_seconds=60) as server2:

    fs_tools = await server2.list_tools()
    print(f"MCP Server is connected. The server offers {len(fs_tools)}")

    for tool in fs_tools:
        print(f'{tool.name} == {tool.description.strip().splitlines()[0]}')


MCP Server is connected. The server offers 14
read_file == Read the complete contents of a file as text. DEPRECATED: Use read_text_file instead.
read_text_file == Read the complete contents of a file from the file system as text. Handles various text encodings and provides detailed error messages if the file cannot be read. Use this tool when you need to examine the contents of a single file. Use the 'head' parameter to read only the first N lines of a file, or the 'tail' parameter to read only the last N lines of a file. Operates on the file as text regardless of extension. Only works within allowed directories.
read_media_file == Read a file and return it as a base64-encoded content block with its MIME type. Image and audio files are returned as image/audio content; any other file type is returned as an embedded resource. Only works within allowed directories.
read_multiple_files == Read the contents of multiple files simultaneously. This is more efficient than reading files one by o

### Filesystem Agent(with MCP Server)

#### Just saying You are a file assistant. You have filesystem tools (vai and MCP Server) is not enough for a smaller model like this you need a robust agent prompt.
* Robust agent prompt means we want the agent to be aware of all information it needs to then make a decision. If I just say the above it doesnt know what directory it has access to and you need to specify everything. 
* If you do not then the onus is one the user and that is not fair, or will not work most time. 

In [7]:
FILES_AGENT_PROMPT = """You are a file assistant. You work inside the directory {SANDBOX_DIR}.
Always use FULL paths under {SANDBOX_DIR} (e.g. {SANDBOX_DIR}/notes.txt).
You have filesystem tools (via an MCP server) to list, read, search, write, and edit files.
When asked about files, first list or read what's there, then act based on what you actually
find. Be concise, and tell the user exactly which files you read or changed. """



In [8]:
#reading in files:

async with MCPServerStdio(
    name = 'Filesystem Server',
    params = filesystem_params,
    client_session_timeout_seconds=60) as server2:
    
    files_agent = Agent(
        name = "Files Agent",
        instructions= FILES_AGENT_PROMPT,
        model= 'gpt-4.1-mini',
        mcp_servers= [server2]
    )

    result = await Runner.run(
        files_agent,
        input = 'What files are in my folder, and what is each one about? Give me a one-line summary per file.',
        max_turns= 20
    )

    print(f'Last Agent:{result.last_agent.name}')
    print('------')
    print(result.final_output)

Last Agent:Files Agent
------
Your folder contains two files:

1. notes.txt - Contains kickoff notes for Project Falcon, a tech news and discussion community featuring AI avatars as commentators.
2. todo.md - A to-do list outlining tasks related to the project launch, such as confirming budget, drafting announcements, and team coordination.


#### Search files:
* mini challenge:
    * #Hints:
#1. Check the traces
#2. Check function definitions
#3. Make input prompt more explicit

In [9]:
#search in files:

async with MCPServerStdio(
    name = 'Filesystem Server',
    params = filesystem_params,
    client_session_timeout_seconds=60) as server2:
    
    files_agent = Agent(
        name = "Files Agent",
        instructions= FILES_AGENT_PROMPT,
        model= 'gpt-4.1-mini',
        mcp_servers= [server2]
    )

    result = await Runner.run(
        files_agent,
        #input = 'Search for files that contain the word Budget.' #this one says missing files b/c it uses the search files tool instead of the tool that gives tool contents,
        input = "Search for files that contain the word Budget (in the files contents)",
        max_turns= 20
    )

    print(f'Last Agent:{result.last_agent.name}')
    print('------')
    print(result.final_output)

Last Agent:Files Agent
------
There are no files containing the word "Budget" in their contents in the current directory. Would you like me to search for the word "Budget" in file names instead?


#### Launching both servers and having the agent have access to both

### CHALLENGE: Two servers, one agent
* Give a single agent both the fetch server and the filesystem server. 
* Then ask it to a fetch a specific web page (your choice) and save a summary of it into the sandobx folder as a new .md file

In [10]:
SUMMARIZER_AGENT_PROMPT = """You are a research assistant. You work inside the directory {SANDBOX_DIR}.

You have two MCP servers: filesystem (list/read/search/write files etc) and fetch(get web url data).
If user asks for some information, use fetch to ge that page. Summarize it and save as an md file.
 """

In [11]:
#params for filesystem server: npx and -y, github url, and directory
filesystem_params = {
    'command' : 'npx',
    'args': ["-y",
        "@modelcontextprotocol/server-filesystem",
        SANDBOX_DIR]
}

In [12]:
#fetch system params: uvx is command and args is ['mcp-server-fetch']
fetch_server_params = {
    'command':'uvx',
    'args':['mcp-server-fetch']
}

In [19]:
async with MCPServerStdio(
    name = 'Filesystem Server',
    params = filesystem_params,
    client_session_timeout_seconds=60) as filesystem_mcp_server:

    async with MCPServerStdio(
    name = 'Fetch Server',
    params= fetch_server_params,
    client_session_timeout_seconds= 60) as fetch_server:
        
        #define agent
        summarizer_agent = Agent(
            name= 'Summarizer Agent',
            instructions= SUMMARIZER_AGENT_PROMPT,
            model = 'gpt-4.1-mini',
            mcp_servers= [filesystem_mcp_server, fetch_server]
        )

        #run agent
        with trace('Summarizer Agent Run', group_id= 'Server Summarizer Agent Flow'):
            result = await Runner.run(
                summarizer_agent,
                input = 'Read the Project Falcon notes to understand what we are building. Then fetch information from this webpage https://www.bbc.com/news/live/c6y8mjmnpxv0t and complete the outstanding research todo item. Write your findings in a competitve teardown in the project folder.',
                max_turns= 30
            )

print(f'Last Agent:{result.last_agent.name}')
print('------')
print(result.final_output)       





Last Agent:Summarizer Agent
------
I could not find the file named "Project Falcon notes" in the project folder. Could you please upload the file or provide the notes so I can understand what we are building? Then I can proceed with fetching information from the webpage and completing the research task.


#### Errors to debug

* The agent doesnt actually read the files, but notes that it can see the files
* The agent announces that it will do the task, but doesnt actually execute the task
* The agent creates the teardown but it doesnt save it to a file

In [ ]:
SUMMARIZER_AGENT_PROMPT = SUMMARIZER_AGENT_PROMPT + """
Always start by listing the direc


"""
